In [4]:
%pip install basemap

import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap  # or import cartopy.crs as ccrs, cartopy.feature as cfeature
import pandas as pd

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 569 kB 4.6 MB/s eta 0:00:01
     |████████████████████████████████| 53 kB 5.8 MB/s  eta 0:00:01
     |████████████████████████████████| 7.5 MB 7.5 MB/s eta 0:00:01
     |████████████████████████████████| 30.5 MB 13.0 MB/s eta 0:00:01
     |████████████████████████████████| 46 kB 4.9 MB/s eta 0:00:01
     |████████████████████████████████| 4.9 MB 5.8 MB/s eta 0:00:01
  Attempting uninstall: packaging
    Found existing installation: packaging 24.2
    Uninstalling packaging-24.2:
      Successfully uninstalled packaging-24.2
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.9.4
    Uninstalling matplotlib-3.9.4:
      Successfully uninstalled matplotlib-3.9.4
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use update

In [ ]:

#pip install tifffile


SyntaxError: invalid syntax (2650537429.py, line 1)

In [ ]:
import os
import glob
import imageio
import numpy as np
from PIL import Image
#import tifffile

ModuleNotFoundError: No module named 'tifffile'

In [ ]:
%pip install imageio
%pip install basemap

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import imageio
import os

In [3]:


def create_lst_animation_single_csv_date(csv_file, output_gif, date_column, fps=5):
    """
    Creates an animation of LST data from a single CSV file with a date column.

    Args:
        csv_file (str): Path to the CSV file.
        output_gif (str): Path to save the output GIF animation.
        date_column (str): Name of the column representing dates.
        fps (int): Frames per second for the animation.
    """

    try:
        df = pd.read_csv(csv_file)
    except FileNotFoundError:
        print(f"CSV file not found: {csv_file}")
        return

    image_files = []
    date_steps = df[date_column].unique()

    for i, date_step in enumerate(date_steps):
        try:
            date_df = df[df[date_column] == date_step]
            lon = date_df['x'].values  # Use 'longitude' column
            lat = date_df['y'].values   # Use 'latitude' column
            lst = date_df['avg_LST'].values        # Use 'avg LST' column
            

            plt.figure(figsize=(8, 6))
            plt.scatter(lon, lat, c=lst, cmap='viridis', s=20)
            plt.colorbar(label='Land Surface Temperature (°C)')
            plt.title(f'LST on {date_step}')
            plt.xlabel('Longitude')
            plt.ylabel('Latitude')

            image_file = f'frame_{i+1:03d}.png'
            plt.savefig(image_file)
            plt.close()

            image_files.append(image_file)

        except Exception as e:
            print(f"Error processing date {date_step}: {e}")
            continue

    if image_files:
        images = [imageio.imread(file) for file in image_files]
        imageio.mimsave(output_gif, images, fps=fps)
        print(f"Animation saved to {output_gif}")

        # Clean up temporary image files
        for file in image_files:
            os.remove(file)
    else:
        print("No images to create animation.")




In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import imageio
import os
from matplotlib.colors import LinearSegmentedColormap
import numpy as np

def create_lst_animation_single_csv_date_shades(csv_file, output_gif, date_column, fps=5):
    """
    Creates an animation of LST data from a single CSV file with a date column, using custom color shades.

    Args:
        csv_file (str): Path to the CSV file.
        output_gif (str): Path to save the output GIF animation.
        date_column (str): Name of the column representing dates.
        fps (int): Frames per second for the animation.
    """

    try:
        df = pd.read_csv(csv_file)
    except FileNotFoundError:
        print(f"CSV file not found: {csv_file}")
        return

    image_files = []
    date_steps = df[date_column].unique()

    for i, date_step in enumerate(date_steps):
        try:
            date_df = df[df[date_column] == date_step]
            lon = date_df['x'].values
            lat = date_df['y'].values
            lst = date_df['avg_LST'].values

            # Custom color mapping with shades
            bins = [0, 25, 30, np.inf]  # Temperature ranges
            purple_cmap = LinearSegmentedColormap.from_list("purple_shades", ["#9370DB", "#4B0082"])
            yellow_cmap = LinearSegmentedColormap.from_list("yellow_shades", ["#FFD700", "#FFA500"])
            red_cmap = LinearSegmentedColormap.from_list("red_shades", ["#FA8072", "#8B0000"])

            # Normalize LST data to bins
            lst_normalized = np.digitize(lst, bins) - 1

            plt.figure(figsize=(8, 6))

            # Apply colormap based on bins
            for j in range(len(lon)):
                if lst_normalized[j] == 0:
                    plt.scatter(lon[j], lat[j], c=purple_cmap(lst[j]/25), s=20) #normalize lst for cmap.
                elif lst_normalized[j] == 1:
                    plt.scatter(lon[j], lat[j], c=yellow_cmap((lst[j]-25)/5), s=20) #normalize lst for cmap.
                else:
                    plt.scatter(lon[j], lat[j], c=red_cmap((lst[j]-30)/(max(lst)-30)), s=20) #normalize lst for cmap.

            # Create custom colorbar (with text labels)
            cbar = plt.colorbar(ticks=[12.5,27.5, max(lst)-((max(lst)-30)/2)])
            cbar.ax.set_yticklabels(['<20', '<30', '>30'])

            plt.title(f'LST on {date_step}')
            plt.xlabel('Longitude')
            plt.ylabel('Latitude')

            image_file = f'frame_{i+1:03d}.png'
            plt.savefig(image_file)
            plt.close()

            image_files.append(image_file)

        except Exception as e:
            print(f"Error processing date {date_step}: {e}")
            continue

    if image_files:
        images = [imageio.imread(file) for file in image_files]
        imageio.mimsave(output_gif, images, fps=fps)
        print(f"Animation saved to {output_gif}")

        # Clean up temporary image files
        for file in image_files:
            os.remove(file)
    else:
        print("No images to create animation.")



In [18]:
# Example for shades :
csv_file = "/Users/varshap/Downloads/ST5188/Data/Final/CHANGI_long.csv"  
output_gif = "lst_animation_custom_shades.gif"
date_column = "period"
create_lst_animation_single_csv_date_shades(csv_file, output_gif, date_column)

/var/folders/xp/9gp1l50s3ksg8hvg7ydc3vjr0000gn/T/ipykernel_73812/2516846881.py:51: UserWarning: *c* argument looks like a single numeric RGB or RGBA sequence, which should be avoided as value-mapping will have precedence in case its length matches with *x* & *y*.  Please use the *color* keyword-argument or provide a 2D array with a single row if you intend to specify the same RGB or RGBA value for all points.
  plt.scatter(lon[j], lat[j], c=yellow_cmap((lst[j]-25)/5), s=20) #normalize lst for cmap.
/var/folders/xp/9gp1l50s3ksg8hvg7ydc3vjr0000gn/T/ipykernel_73812/2516846881.py:49: UserWarning: *c* argument looks like a single numeric RGB or RGBA sequence, which should be avoided as value-mapping will have precedence in case its length matches with *x* & *y*.  Please use the *color* keyword-argument or provide a 2D array with a single row if you intend to specify the same RGB or RGBA value for all points.
  plt.scatter(lon[j], lat[j], c=purple_cmap(lst[j]/25), s=20) #normalize lst for cm

Animation saved to lst_animation_custom_shades.gif


In [5]:

csv_file = "/Users/varshap/Downloads/ST5188/Data/Final/CHANGI_long.csv"  
output_gif = "lst_animation_changi.gif"
date_column = "period" 
create_lst_animation_single_csv_date(csv_file, output_gif, date_column)

/var/folders/xp/9gp1l50s3ksg8hvg7ydc3vjr0000gn/T/ipykernel_62511/2790383113.py:47: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images = [imageio.imread(file) for file in image_files]


Animation saved to lst_animation_changi.gif


/var/folders/xp/9gp1l50s3ksg8hvg7ydc3vjr0000gn/T/ipykernel_62511/2790383113.py:47: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images = [imageio.imread(file) for file in image_files]


Animation saved to lst_animation_jurong_W.gif


In [9]:
csv_file = "/Users/varshap/Downloads/ST5188/Data/Final/JURONG EAST_long.csv"  
output_gif = "lst_animation_jurong_E.gif"
date_column = "period" 
create_lst_animation_single_csv_date(csv_file, output_gif, date_column)

/var/folders/xp/9gp1l50s3ksg8hvg7ydc3vjr0000gn/T/ipykernel_62511/2790383113.py:47: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images = [imageio.imread(file) for file in image_files]


Animation saved to lst_animation_jurong_E.gif


In [13]:
csv_file = "/Users/varshap/Downloads/ST5188/Data/Final/JURONG WEST_long.csv"  
output_gif = "lst_animation_jurong_W.gif"
date_column = "period" 
create_lst_animation_single_csv_date(csv_file, output_gif, date_column)

/var/folders/xp/9gp1l50s3ksg8hvg7ydc3vjr0000gn/T/ipykernel_62511/2790383113.py:47: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images = [imageio.imread(file) for file in image_files]


Animation saved to lst_animation_jurong_W.gif


In [16]:
#Run this after each example
create_lst_animation_single_csv_date(csv_file, output_gif, date_column, fps=2)

NameError: name 'create_lst_animation_single_csv_date' is not defined